In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pickle
from pathlib import Path

DATA_DIR = Path("../data/cifar-10-python/cifar-10-batches-py/")

def unpickle(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="bytes")

def load_batch(path):
    d = unpickle(path)
    x = d[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # -> (N, 32, 32, 3)
    y = np.array(d[b"labels"])
    return x, y

meta = unpickle(DATA_DIR / "batches.meta")
class_names = [n.decode() for n in meta[b"label_names"]]

xs, ys = zip(*[load_batch(DATA_DIR / f"data_batch_{i}") for i in range(1, 6)])
x_all, y_all = np.concatenate(xs), np.concatenate(ys)
x_test, y_test = load_batch(DATA_DIR / "test_batch")

print(x_all.shape, y_all.shape, x_test.shape)   # (50000,32,32,3) (50000,) (10000,32,32,3)
print(np.bincount(y_all))                        # her sınıf 5000 olmalı

(50000, 32, 32, 3) (50000,) (10000, 32, 32, 3)
[5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]


In [3]:
from sklearn.model_selection import train_test_split

In [4]:
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=0.1, stratify=y_all, random_state=42
)

In [5]:
mean = x_train.reshape(-1, 3).mean(axis=0) / 255.0
std = x_train.reshape(-1, 3).std(axis=0) / 255.0

In [6]:
def normalize(x):
    return (((x / 255.0) - mean) / std).astype("float32")

In [7]:
x_train_n, x_val_n, x_test_n = normalize(x_train), normalize(x_val), normalize(x_test)

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

In [9]:
from tensorflow.keras import layers, models

In [10]:
import tensorflow as tf

In [11]:
def build_baseline():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [12]:
baseline = build_baseline()

In [13]:
history = baseline.fit(x_train_n, y_train, validation_data=(x_val_n, y_val), epochs=20, batch_size=128)

Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 216s 592ms/step - accuracy: 0.5167 - loss: 1.3692 - val_accuracy: 0.6398 - val_loss: 1.0447
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 182s 364ms/step - accuracy: 0.6614 - loss: 0.9743 - val_accuracy: 0.6748 - val_loss: 0.9390
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 104s 296ms/step - accuracy: 0.7171 - loss: 0.8162 - val_accuracy: 0.6878 - val_loss: 0.8991
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 88s 250ms/step - accuracy: 0.7598 - loss: 0.7006 - val_accuracy: 0.7104 - val_loss: 0.8596
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 92s 260ms/step - accuracy: 0.7920 - loss: 0.6063 - val_accuracy: 0.7162 - val_loss: 0.8335
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 277s 642ms/step - accuracy: 0.8255 - loss: 0.5103 - val_accuracy: 0.7124 - val_loss: 0.8530
Epoch 7/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 228s 544ms/step - accuracy: 0.8550 - loss: 0.4241 - val_accuracy: 0.7294 - val_loss: 0.8667
Epoch 8/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 165s 468ms/step - accuracy: 0.8827 - l

In [14]:
test_loss, test_acc = baseline.evaluate(x_test_n, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

Test accuracy: 0.7008


In [15]:
y_pred = baseline.predict(x_test_n).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=class_names))
print(confusion_matrix(y_test, y_pred))

313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 52ms/step
              precision    recall  f1-score   support

    airplane       0.79      0.69      0.74      1000
  automobile       0.77      0.86      0.81      1000
        bird       0.64      0.55      0.59      1000
         cat       0.46      0.58      0.51      1000
        deer       0.71      0.62      0.66      1000
         dog       0.57      0.64      0.61      1000
        frog       0.72      0.81      0.76      1000
       horse       0.82      0.72      0.77      1000
        ship       0.85      0.78      0.81      1000
       truck       0.78      0.76      0.77      1000

    accuracy                           0.70     10000
   macro avg       0.71      0.70      0.70     10000
weighted avg       0.71      0.70      0.70     10000

[[695  38  41  48  16  14  14  15  64  55]
 [ 13 859   7  15   4  12   9   3  10  68]
 [ 54   9 550 109  61  71  94  27  13  12]
 [ 14  12  48 579  42 196  63  20  11  15]
 [ 17  11  78  98 623  47 

In [17]:
baseline.save("../model/baseline.keras")